# Week 14 - Final Project Development

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 14 - Final Project Development**

---

## Session topics (from the course outline)

- Project scoping and dataset selection
- Building the analysis
- Preparing the report

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 4** | Use Matplotlib and Seaborn to create charts for effective presentation |
| **CLO 5** | Perform exploratory data analysis (EDA) to uncover insights |
| **CLO 8** | Evaluate the effectiveness of different visualization methods |
| **CLO 9** | Design interactive dashboards or visual reports to present data findings |
| **CLO 10** | Create a complete end-to-end data analysis project |

## Reading

| Topic | Source |
|---|---|
| Case studies | Pfaffelhuber §10 |
| Week 13 of this course | `13_case_study.ipynb` — the template your report follows |

## What this session is

A working session. By the end of it you should have a question, a dataset you have checked, a
plan that fits in the time available, and a project notebook with its sections already in place.

The failure mode this session exists to prevent is the common one: three weeks of effort on a
question the data were never able to answer. That outcome is decided in the first hour, not the
last, and the checks in Part 2 are the ones that catch it.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from scipy import stats
from statsmodels.stats.power import TTestIndPower

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 — The project brief

---

## 1.1 What you are producing

Two deliverables.

| Deliverable | Form | Due |
|---|---|---|
| **The project notebook** | One `.ipynb`, named `final_project.ipynb`, in your course repository | End of Week 15 |
| **The presentation** | 10 minutes plus 5 for questions, in the Week 15 session | Week 15 session |

The notebook is a report that happens to be executable. Somebody who has never seen your data
should be able to open it, read it top to bottom, understand what you asked and what you found,
and re-run it to get the same numbers.

**The assessment weighting for this course is Attendance and participation 10%, Mid-Term 40%,
Final 50%.** The point allocations below are indicative — they describe how the work is judged,
and your instructor will say how they map onto the final mark.

---

## 1.2 The rubric

| Component | Points | What earns them |
|---|---|---|
| **Question and scope** | 10 | A specific, measurable, time-bounded question tied to a decision. Stated before the analysis, not reverse-engineered from it |
| **Data audit and cleaning** | 20 | The six audit questions asked and answered. Cleaning written as code, printed, and justified. Faults reported, not silently dropped |
| **Exploration and visualization** | 20 | Charts that answer something. Correct chart type, labelled axes, honest scales. Each figure followed by what it shows |
| **Statistical analysis** | 15 | At least one quantified comparison with an interval or effect size. Assumptions stated. No p-value reported on its own |
| **Findings and recommendation** | 20 | The five-part findings statement from Week 13 §2.8. Limitations named. A recommendation somebody could act on |
| **Reproducibility and code quality** | 10 | Restart-and-run-all works. Seed fixed. Functions with docstrings. No dead cells, no commented-out experiments |
| **Presentation** | 5 | Ten minutes, three findings, no wall of text |

**The findings section carries the most marks and is the one most often left until the last
hour.** Write a first draft of it in Week 14, before the analysis is finished. It will be wrong,
and rewriting it is how you find out what the analysis still needs.

---

## 1.3 Dataset requirements

Your dataset must satisfy all of these.

| Requirement | Minimum | Why |
|---|---|---|
| Rows | 500 | Below this almost nothing is measurable — see §2.3 |
| Columns | 5 | You need something to group by and something to control for |
| A numeric outcome | 1 | The thing your question is about |
| A grouping column **or** a date column | 1 | Otherwise there is nothing to compare |
| Real, and not pre-cleaned | — | 20 of the 100 points are for cleaning |

**Where to get one.** Kaggle, `data.gov`, a national statistics office, a company or NGO you
work with, or data you collect yourself. Download it **before** the session — the course
environment has no internet access during class.

**Datasets that will not be accepted**, because they have no cleaning left in them and every
finding has been published a thousand times: `iris`, `titanic`, `mtcars`, `tips`, the seaborn
built-ins, and any dataset that ships as an example with a tutorial.

**If you use data about identifiable people**, remove direct identifiers before you start, do
not publish the raw file in your repository, and say in the notebook what you removed.

---

## 1.4 The three ways a project goes wrong

**Too big.** "Analyse the effect of climate change on Cambodian agriculture." Three months of
work, and you have four weeks. The fix is always the same: add a place, a period and a single
outcome. "Did rice yields in Battambang province change between 2015 and 2024, relative to the
national average?"

**Too small.** "Plot the distribution of ages in this dataset." That is one cell, not a project.
If the answer is a single number with no comparison in it, the question is not big enough.

**Not a question.** "Explore the sales dataset." There is no answer, so there is no way to be
finished, and no way to be wrong. Every project that ends as a tour of the columns started here.

The test that catches all three: **write the sentence "depending on the answer, somebody would
___".** If you cannot finish it, the question is not ready.

---

## 1.5 The report structure

The same as Week 13's notebook, because that is what it was built to be.

```
   1  QUESTION        what you asked, why, and what it would change
   2  THE DATA        source, what a row is, how it was obtained
   3  AUDIT           the six questions, answered
   4  CLEANING        as code, with a printed ledger of what changed
   5  EXPLORATION     the charts that answer the sub-questions
   6  ANALYSIS        one quantified comparison, with uncertainty
   7  FINDINGS        five sentences
   8  LIMITATIONS     what this analysis cannot tell you
   9  REFERENCES      data source, and anything you read
```

Section 2.5 generates this as a notebook file you can start from.

---

## 1.6 The schedule

You have this session, the week between, and the Week 15 session.

| Checkpoint | By | What must exist |
|---|---|---|
| **1 — Scoped** | End of today | Question written, dataset checked with `assess_dataset`, plan validated |
| **2 — Cleaned** | Three days later | Audit and cleaning sections complete, panel built, one exploratory figure |
| **3 — Analysed** | Two days before Week 15 | The comparison run, the first draft of the findings written |
| **Submission** | End of Week 15 | Restart and run all, commit, push |

The most common scheduling mistake is leaving the cleaning until the analysis needs it.
Cleaning takes longer than you expect every single time, and it is worth 20 points on its own.

---

## 1.7 Attribution and honesty

- **Cite the data source** in section 9, with a URL and the date you downloaded it.
- **Cite anything you read** that shaped the analysis, including tutorials and forum answers.
- **Code you did not write** must be marked as such, with its origin, whether it came from a
  book, a website or a language model. Using it is allowed. Presenting it as your own is not.
- **Report what you tried**, not only what worked. An analysis that ran six comparisons and
  reports one is the Week 12 §2.7 problem, and it is visible to a marker who reads your code.
- **Never edit the data file by hand.** If a value is wrong, fix it in code so the fix is
  visible and repeatable.

---

---

# Part 2 — Scoping a project

We take one student's situation and work it through: three candidate questions, a dataset,
the two checks that decide whether the pairing can work, and the scaffold to start from.

---

## 2.1 Three candidate questions, scored

The student has a dataset of clinic visits from a district health office. Three questions
occurred to them.

In [ ]:
CRITERIA = ["specific", "measurable", "time_bounded", "actionable"]

candidates = pd.DataFrame(
    [
        {"question": "How is public health in Cambodia changing?",
         "specific": 0, "measurable": 0, "time_bounded": 0, "actionable": 0},
        {"question": "What is the average age of patients at this clinic?",
         "specific": 1, "measurable": 1, "time_bounded": 0, "actionable": 0},
        {"question": "Did afternoon clinic sessions have longer waiting times than morning "
                     "sessions in 2025, and by how much?",
         "specific": 1, "measurable": 1, "time_bounded": 1, "actionable": 1},
    ]
).set_index("question")

candidates["score"] = candidates[CRITERIA].sum(axis=1)
print(candidates.to_string())

print("\nThe test that decides it: finish the sentence")
print('   "depending on the answer, somebody would ______"\n')
for question, score in candidates["score"].items():
    verdict = {0: "no answer exists — it is a topic, not a question",
               2: "one number, no comparison — that is a cell, not a project",
               4: "-> move the afternoon session, or staff it differently"}[score]
    print(f"  score {score}  {verdict}")

The third question is not more ambitious than the first. It is **less** ambitious, and that is
why it is the only one that can be finished. A project is a question small enough to answer
completely and large enough to have a comparison in it.

---

## 2.2 Does the dataset support the question?

Two datasets are generated below. One is the clinic's full 2025 export; the other is a small
pilot extract a different student brought. They are checked the same way.

In [ ]:
def build_clinic_visits(n=1800):
    """A district clinic's 2025 visit log. Stands in for a downloaded CSV."""
    dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(
        rng.integers(0, 365, size=n), unit="D")
    session = rng.choice(["morning", "afternoon"], size=n, p=[0.58, 0.42])
    base = np.where(session == "afternoon", 34.0, 26.0)
    wait = np.clip(rng.normal(base, 13.0, size=n), 2.0, None).round(1)
    wait[rng.choice(n, size=int(0.04 * n), replace=False)] = np.nan
    return pd.DataFrame({
        "visit_id": [f"V{i:05d}" for i in range(1, n + 1)],
        "date": dates,
        "session": session,
        "department": rng.choice(["general", "maternal", "dental", "pharmacy"],
                                 size=n, p=[0.46, 0.24, 0.14, 0.16]),
        "patient_age": np.clip(rng.gamma(6.0, 6.5, size=n), 0, 95).round(0),
        "wait_minutes": wait,
    })


def build_pilot_extract(n=60):
    """A three-week pilot from a different clinic. Too small, and the check should say so."""
    dates = pd.to_datetime("2025-06-01") + pd.to_timedelta(
        rng.integers(0, 20, size=n), unit="D")
    session = rng.choice(["morning", "afternoon", "evening"], size=n, p=[0.5, 0.45, 0.05])
    return pd.DataFrame({
        "visit_id": [f"P{i:03d}" for i in range(1, n + 1)],
        "date": dates,
        "session": session,
        "wait_minutes": np.clip(rng.normal(30.0, 12.0, size=n), 2.0, None).round(1),
    })


visits = build_clinic_visits()
pilot = build_pilot_extract()
print(visits.head().to_string(index=False))
print(f"\nvisits {visits.shape}   pilot {pilot.shape}")

In [ ]:
MIN_ROWS = 500
MIN_GROUP = 10
MIN_SPAN_DAYS = 14


def assess_dataset(df, outcome, group=None, date=None) -> dict:
    """Decide whether a dataset can support a question, before any analysis is done.

    Parameters
    ----------
    df : pandas.DataFrame
        The candidate dataset. Not modified.
    outcome : str
        The numeric column the question is about.
    group : str or None
        The column the comparison is across, if there is one.
    date : str or None
        A datetime column, if the question is time-bounded.

    Returns
    -------
    dict
        n_rows, n_columns, outcome_missing_pct, n_groups, smallest_group,
        span_days, blockers, warnings, verdict.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"expected a DataFrame, got {type(df).__name__}")
    for column in [outcome, group, date]:
        if column is not None and column not in df.columns:
            raise KeyError(f"no column named {column!r}")
    if not pd.api.types.is_numeric_dtype(df[outcome]):
        raise TypeError(f"the outcome {outcome!r} is not numeric")
    if date is not None and not pd.api.types.is_datetime64_any_dtype(df[date]):
        raise TypeError(f"{date!r} is not a datetime column")

    blockers, warnings = [], []
    n_rows, n_columns = df.shape

    if n_rows < MIN_ROWS:
        blockers.append(f"only {n_rows} rows; the brief requires at least {MIN_ROWS}")
    elif n_rows < 2 * MIN_ROWS:
        warnings.append(f"{n_rows} rows is workable but leaves little room for subgroups")
    if n_columns < 5:
        warnings.append(f"only {n_columns} columns; little scope for a control variable")

    missing_pct = float(df[outcome].isna().mean() * 100)
    if missing_pct > 50:
        blockers.append(f"the outcome is {missing_pct:.0f}% missing")
    elif missing_pct > 5:
        warnings.append(f"the outcome is {missing_pct:.1f}% missing; say how you handle it")

    n_groups = smallest_group = None
    if group is not None:
        counts = df[group].value_counts()
        n_groups, smallest_group = int(counts.size), int(counts.min())
        if n_groups < 2:
            blockers.append(f"{group!r} has one value; there is nothing to compare")
        if smallest_group < MIN_GROUP:
            blockers.append(f"the smallest group in {group!r} has {smallest_group} rows")
        if n_groups > 20:
            warnings.append(f"{n_groups} groups is a lot; consider combining them")

    span_days = None
    if date is not None:
        span_days = int((df[date].max() - df[date].min()).days)
        if span_days < MIN_SPAN_DAYS:
            blockers.append(f"the data span {span_days} days; too short for a trend")
        elif span_days < 90:
            warnings.append(f"{span_days} days will not show a seasonal pattern")

    verdict = "stop" if blockers else ("review" if warnings else "go")
    return {"n_rows": n_rows, "n_columns": n_columns,
            "outcome_missing_pct": round(missing_pct, 2),
            "n_groups": n_groups, "smallest_group": smallest_group,
            "span_days": span_days, "blockers": blockers,
            "warnings": warnings, "verdict": verdict}


def print_assessment(report, label):
    print(f"{label}: {report['verdict'].upper()}")
    for key in ["n_rows", "n_columns", "outcome_missing_pct", "n_groups",
                "smallest_group", "span_days"]:
        print(f"    {key:<20} {report[key]}")
    for item in report["blockers"]:
        print(f"    BLOCKER  {item}")
    for item in report["warnings"]:
        print(f"    warning  {item}")
    print()


print_assessment(assess_dataset(visits, "wait_minutes", group="session", date="date"),
                 "clinic 2025 export")
print_assessment(assess_dataset(pilot, "wait_minutes", group="session", date="date"),
                 "three-week pilot")

The pilot is not a small version of the project. It is a different project — and the check said
so in two seconds, before anybody spent a weekend on it.

That is the whole purpose of running this first. The student with the pilot has two options:
collect more data, or change the question to one 60 rows can answer. Finding that out now is
worth more than any technique in this course.

---

## 2.3 What size of effect could this dataset detect?

The dataset passes the row count. That is not the same as being able to answer the question.

Week 12 gave the relationship: with equal groups, $t = d\sqrt{n/2}$. Turn it around and it says
what the **smallest difference you could reliably detect** is, given the sample size and the
spread. Anything smaller than that, your study will miss — as Week 13's case study did.

In [ ]:
def minimum_detectable_effect(sd, n_per_group, alpha=0.05, power=0.80, relative=False):
    """Smallest difference in means a two-group study could reliably detect.

    Uses the normal approximation

        MDE = (z_(1-alpha/2) + z_power) * sd * sqrt(2 / n)

    Parameters
    ----------
    sd : float
        Standard deviation of the outcome within a group. Must be positive.
    n_per_group : int
        Observations in each group. At least 2.
    alpha, power : float
        Significance level and desired power, each strictly between 0 and 1.
    relative : bool
        If True, return the effect in standard deviations (Cohen's d) instead of
        in the units of the outcome.

    Returns
    -------
    float
    """
    if sd <= 0:
        raise ValueError(f"sd must be positive; got {sd}")
    if n_per_group < 2:
        raise ValueError(f"n_per_group must be at least 2; got {n_per_group}")
    for name, value in [("alpha", alpha), ("power", power)]:
        if not 0.0 < value < 1.0:
            raise ValueError(f"{name} must be strictly between 0 and 1; got {value}")

    multiplier = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
    effect = multiplier * np.sqrt(2.0 / n_per_group)
    return float(effect if relative else effect * sd)


usable_visits = visits.dropna(subset=["wait_minutes"])
counts = usable_visits["session"].value_counts()
sd_within = usable_visits.groupby("session")["wait_minutes"].std(ddof=1).mean()
n_smaller = int(counts.min())

mde = minimum_detectable_effect(sd_within, n_smaller)
observed = (usable_visits.loc[usable_visits["session"] == "afternoon", "wait_minutes"].mean()
            - usable_visits.loc[usable_visits["session"] == "morning", "wait_minutes"].mean())

print(f"within-session sd            {sd_within:6.2f} minutes")
print(f"smaller group                {n_smaller:6d} visits")
print(f"smallest detectable effect   {mde:6.2f} minutes  (80% power, alpha 0.05)")
print(f"                             {minimum_detectable_effect(sd_within, n_smaller, relative=True):6.3f} in sd units")
print(f"\nactually observed            {observed:6.2f} minutes")
print(f"\nThe question is answerable: the difference the clinic cares about is several times")
print("larger than the smallest one this dataset could have found.")

In [ ]:
# The same calculation across sample sizes — the table to consult when scoping.
sizes = [30, 60, 120, 250, 500, 1000]
planning = pd.DataFrame({
    "n per group": sizes,
    "MDE (minutes)": [round(minimum_detectable_effect(sd_within, n), 2) for n in sizes],
    "MDE (in sd)": [round(minimum_detectable_effect(sd_within, n, relative=True), 3)
                    for n in sizes],
})
print(planning.to_string(index=False))

# statsmodels does the same thing with the t distribution rather than the normal.
exact = TTestIndPower().solve_power(nobs1=n_smaller, alpha=0.05, power=0.80,
                                    ratio=1.0, alternative="two-sided")
print(f"\nnormal approximation, n = {n_smaller}:  d = "
      f"{minimum_detectable_effect(sd_within, n_smaller, relative=True):.4f}")
print(f"statsmodels, t-based:              d = {exact:.4f}")
print("The approximation is close at this sample size and optimistic at small ones.")

fig, ax = plt.subplots(figsize=(9, 3.8))
grid = np.arange(20, 1001, 5)
ax.plot(grid, [minimum_detectable_effect(sd_within, n) for n in grid],
        color=PALETTE[0], lw=2)
ax.axvline(n_smaller, color=PALETTE[2], ls="--", lw=1.2)
ax.axhline(abs(observed), color=PALETTE[3], ls="--", lw=1.2)
ax.text(n_smaller, ax.get_ylim()[1] * 0.9, f"  your n = {n_smaller}", fontsize=9)
ax.text(1000, abs(observed), f"observed difference  ", fontsize=9, ha="right", va="bottom")
ax.set_title("Smallest detectable difference against sample size", loc="left")
ax.set_xlabel("observations per group")
ax.set_ylabel("minutes")
plt.tight_layout()
plt.show()

Read the curve the way you would read a price list. It falls steeply to about 250 per group and
then flattens, because precision improves with $\sqrt{n}$. Going from 30 to 120 buys you a great
deal; going from 500 to 1000 buys you very little.

**Use it before you commit, not after.** If the difference you care about sits below the curve
at the sample size you have, the study cannot answer the question, and there are only three
responses: get more data, accept a coarser question, or measure something with less noise in it.

---

## 2.4 The plan, and a check on it

In [ ]:
REQUIRED_PLAN_KEYS = ["question", "decision", "dataset", "source", "outcome",
                      "comparison", "rows", "risks"]


def validate_plan(plan) -> list:
    """Check a project plan for the things that are cheap to fix now and expensive later.

    Returns
    -------
    list of str
        The problems found. An empty list means the plan is ready.
    """
    if not isinstance(plan, dict):
        raise TypeError(f"expected a dict, got {type(plan).__name__}")

    problems = []
    for key in REQUIRED_PLAN_KEYS:
        if key not in plan:
            problems.append(f"missing key: {key!r}")
        elif isinstance(plan[key], str) and not plan[key].strip():
            problems.append(f"{key!r} is empty")
    if problems:
        return problems

    if not plan["question"].strip().endswith("?"):
        problems.append("'question' must be a question, ending in '?'")
    if len(plan["question"].split()) < 8:
        problems.append("'question' is too short to be specific; name what, where and when")
    if not any(word in plan["decision"].lower()
               for word in ["would", "will", "should", "could", "decide"]):
        problems.append("'decision' must say what somebody would DO differently")
    if plan["rows"] < 500:
        problems.append(f"'rows' is {plan['rows']}; the brief requires at least 500")
    if len(plan["risks"]) < 2:
        problems.append("list at least two risks; every project has more than one")
    if "http" not in plan["source"] and "collected" not in plan["source"].lower():
        problems.append("'source' should be a URL, or say how the data were collected")
    return problems


example_plan = {
    "question": "Did afternoon clinic sessions have longer waiting times than morning "
                "sessions during 2025, and by how much?",
    "decision": "If afternoons are more than ten minutes slower, the clinic would move two "
                "staff to the afternoon rota.",
    "dataset": "District clinic visit log, 2025",
    "source": "https://example-health-office.gov.kh/open-data/visits-2025.csv",
    "outcome": "wait_minutes",
    "comparison": "session (morning against afternoon)",
    "rows": len(visits),
    "risks": [
        "waiting time is recorded by hand and may be rounded to five minutes",
        "department mix differs between sessions and may explain the gap",
        "4% of waiting times are missing and the reason is not recorded",
    ],
}

problems = validate_plan(example_plan)
print("problems found:", problems if problems else "none — the plan is ready")
print()
for key, value in example_plan.items():
    print(f"  {key:<12} {value}")

The third risk in that list is the interesting one, and it is the kind of thing this exercise is
for. *Department mix differs between sessions and may explain the gap* is a confounder, named
before the analysis starts. Week 13 showed what happens when one is found afterwards.

Writing the risks down at this stage costs five minutes and changes what you check in the audit.

---

## 2.5 Generate your project scaffold

A `.ipynb` file is JSON. Nothing more. The function below writes the nine sections of §1.5 into
a notebook you can open and start filling in.

In [ ]:
SECTION_TEMPLATE = [
    ("1. Question", "What did you ask, why, and what would the answer change?\n\n"
                    "State the question in one sentence, ending in a question mark. Then the "
                    "sentence: *depending on the answer, somebody would ...*"),
    ("2. The data", "Where it came from, with a URL and the date you downloaded it. What one "
                    "row represents. How it was collected, and by whom."),
    ("3. Audit", "The six questions from Week 13 §1.3: size and types, missingness, "
                 "duplicates, categories, ranges, and the time axis. Show the checks."),
    ("4. Cleaning", "Every step as code, with a printed ledger of what changed. Justify each "
                    "decision, especially anything you removed."),
    ("5. Exploration", "The charts that answer your sub-questions. Each figure followed by one "
                       "paragraph saying what it shows."),
    ("6. Analysis", "One quantified comparison, with a confidence interval or an effect size. "
                    "State the assumptions. Do not report a p-value on its own."),
    ("7. Findings", "Five sentences: what was measured, how much, how certain, compared with "
                    "what, and so what."),
    ("8. Limitations", "What this analysis cannot tell you, and what would be needed to answer "
                       "it properly."),
    ("9. References", "Data source, and anything you read that shaped the analysis. Mark any "
                      "code you did not write yourself."),
]


def write_project_template(path="final_project.ipynb", author="YOUR NAME"):
    """Write an empty project notebook with the nine required sections.

    A .ipynb file is a JSON document with a list of cells. Building one by hand takes
    no libraries at all, which is worth knowing the first time a notebook will not open.
    """
    cells = [{
        "cell_type": "markdown", "metadata": {},
        "source": (f"# Final Project — {author}\n\n"
                   "**Data Analysis and Visualization** — "
                   "Paññāsāstra University of Cambodia\n\n"
                   "> Replace this line with your question, in one sentence.\n\n---\n"),
    }, {
        "cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [],
        "source": ("import numpy as np\nimport pandas as pd\n"
                   "import matplotlib.pyplot as plt\nimport seaborn as sns\n\n"
                   "RANDOM_SEED = 42\nrng = np.random.default_rng(RANDOM_SEED)\n\n"
                   "PALETTE = [\"#4C72B0\", \"#DD8452\", \"#55A868\", \"#C44E52\"]\n"
                   "sns.set_palette(PALETTE)\n"
                   "pd.set_option(\"display.max_columns\", 30)\n"),
    }]

    for heading, guidance in SECTION_TEMPLATE:
        cells.append({"cell_type": "markdown", "metadata": {},
                      "source": f"---\n\n## {heading}\n\n*{guidance}*\n"})
        cells.append({"cell_type": "code", "metadata": {}, "execution_count": None,
                      "outputs": [], "source": f"# {heading}\n"})

    notebook = {
        "cells": cells,
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python",
                           "name": "python3"},
            "language_info": {"name": "python"},
        },
        "nbformat": 4, "nbformat_minor": 5,
    }

    Path(path).write_text(json.dumps(notebook, indent=1), encoding="utf-8")
    return Path(path)


written = write_project_template("final_project.ipynb", author="Sokha Chan")
print(f"wrote {written.name}  ({written.stat().st_size:,} bytes)")

check = json.loads(written.read_text(encoding="utf-8"))
print(f"  {len(check['cells'])} cells")
print(f"  sections: {[h for h, _ in SECTION_TEMPLATE]}")
print("\nOpen it in VS Code, select your venv kernel, and start at section 1.")

---

## 2.6 Two findings sections, marked

The same analysis, written up two ways. Both are honest; only one is worth 20 points.

**Submission A**

> Our analysis shows that afternoon sessions have longer waiting times than morning sessions.
> This is statistically significant (p < 0.001). The clinic should therefore address afternoon
> staffing. See the boxplot in section 5 for details.

**Submission B**

> Afternoon sessions waited a mean of 7.5 minutes longer than morning sessions (95% CI 6.2 to
> 8.7 minutes; d = 0.59; n = 1,728 visits during 2025). The gap holds within all four
> departments, ranging from +6.6 to +10.2 minutes, so it is not an artefact of the afternoon
> department mix. It is 28% of the mean morning wait of 26.4 minutes and would be obvious to a
> patient. The clinic's own threshold for acting is ten minutes, and the whole interval lies
> below it — so this supports reviewing the afternoon rota rather than immediately restaffing
> it. The main limitation is that waiting times are recorded by hand and cluster on
> five-minute values, which widens the true uncertainty beyond the interval above.

| Criterion | A | B |
|---|---|---|
| What was measured, in the units of the problem | no — "longer" | yes — 7.5 minutes |
| How much | no | yes |
| How certain | p-value only | interval, effect size, sample size |
| Compared with what | nothing | the morning mean, the four departments, and the clinic's threshold |
| So what | vague | a specific action, and why not a stronger one |
| Confounders addressed | not mentioned | department mix checked and ruled out |
| Limitations | none | the recording granularity, named |
| **Indicative mark** | **6 / 20** | **19 / 20** |

Submission A is not wrong. Every sentence in it is true. It is simply not usable: nobody reading
it can decide anything, and "statistically significant" is doing work it cannot do — Week 12
§2.6.

Notice how much of B comes from outside the statistics: the clinic's ten-minute threshold, the
knowledge that times are recorded by hand, the decision to check the department mix. That is the
Week 13 lesson again.

---

## 2.7 The risk register

Five minutes, written before the analysis. Each row is something that could make the answer
wrong, and what you will do about it.

| Risk | Likelihood | What you will do |
|---|---|---|
| Waiting times rounded to 5 minutes by hand | high | Check the value distribution in the audit; report the granularity as a limitation |
| Department mix differs between sessions | medium | Compare within department as well as overall |
| 4% of waiting times missing, reason unknown | high | Check whether missingness differs by session; if it does, say so and do not fill |
| The dataset is one clinic, so the result may not generalise | certain | State it in limitations; do not write "clinics" when you mean "this clinic" |
| Something changed mid-year that nobody mentioned | medium | Plot the time axis, as in Week 13 §2.4 |

The last row belongs in every risk register you will ever write.

---

---

# Part 3 — Guided workshop (in class)

Four tasks. Tasks 1 to 3 rehearse the checks on a dataset supplied here; task 4 applies them to
**your own** project, which is the one that matters.

### Task 1 — Assess a dataset

A student has a household survey and wants to compare monthly food spending across regions.

**Problem.** Run `assess_dataset` on `survey` with outcome `spend_usd`, group `region` and date
`interview_date`. Assign three variables:

| Variable | Contents |
|---|---|
| `survey_report` | the dictionary `assess_dataset` returns |
| `survey_verdict` | its `"verdict"` value |
| `first_blocker` | the first entry of its `"blockers"` list, or `None` if there are none |

**Hints**

- `assess_dataset(survey, "spend_usd", group="region", date="interview_date")`.
- `print_assessment(survey_report, "survey")` prints it in the format used above.
- Reading the blocker matters more than producing it.

**Expected output.** A verdict of `stop`, one blocker, then `Task 1 complete`.

In [ ]:
def build_survey(n=900):
    """A household survey with one region far too small to compare."""
    sizes = {"Phnom Penh": 300, "Kandal": 195, "Siem Reap": 160, "Battambang": 130,
             "Kampot": 100, "Ratanakiri": 5, "Mondulkiri": 4, "Preah Vihear": 6}
    region = np.repeat(list(sizes), list(sizes.values()))
    order = rng.permutation(n)
    region = region[order]

    base = pd.Series(region).map({
        "Phnom Penh": 128.0, "Kandal": 104.0, "Siem Reap": 96.0, "Battambang": 92.0,
        "Kampot": 88.0, "Ratanakiri": 74.0, "Mondulkiri": 78.0, "Preah Vihear": 80.0,
    }).to_numpy()

    return pd.DataFrame({
        "household_id": [f"H{i:04d}" for i in range(1, n + 1)],
        "interview_date": pd.to_datetime("2025-03-01") + pd.to_timedelta(
            rng.integers(0, 120, size=n), unit="D"),
        "region": region,
        "household_size": rng.integers(1, 9, size=n),
        "spend_usd": np.clip(rng.normal(base, 26.0, size=n), 10.0, None).round(2),
    })


survey = build_survey()
print(survey.head().to_string(index=False))
print(f"\n{survey.shape[0]} rows, {survey.shape[1]} columns")
print(survey["region"].value_counts().to_string())

In [ ]:
# TODO 1: assess the survey dataset.
survey_report = None
survey_verdict = None
first_blocker = None

if survey_report is not None:
    print_assessment(survey_report, "household survey")

In [ ]:
# Verification for Task 1.
truth = assess_dataset(survey, "spend_usd", group="region", date="interview_date")
problems = []

if survey_report is None:
    problems.append("survey_report is still None")
elif not isinstance(survey_report, dict):
    problems.append(f"survey_report should be a dict, got {type(survey_report).__name__}")
elif survey_report != truth:
    differing = [k for k in truth if survey_report.get(k) != truth[k]]
    problems.append(f"survey_report does not match; differing keys: {differing}")
else:
    if survey_verdict != truth["verdict"]:
        problems.append(f"survey_verdict should be {truth['verdict']!r}, got {survey_verdict!r}")
    expected_blocker = truth["blockers"][0] if truth["blockers"] else None
    if first_blocker != expected_blocker:
        problems.append(f"first_blocker should be {expected_blocker!r}, got {first_blocker!r}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 1 complete. Verdict: {survey_verdict}.")
    print(f"Blocker: {first_blocker}")
    print("\n900 rows and four months of data, and the project is still blocked — by three")
    print("regions with fewer than ten households between them.")

### Task 2 — Fix it, without throwing data away

The blocker is that three regions are far too small to compare. Deleting them loses 15
households; combining them keeps every row and makes the comparison possible.

**Problem.** Build `survey_fixed`: a copy of `survey` in which every region with fewer than
`MIN_GROUP` households is relabelled `"Other"`. Then assign `fixed_report` and
`fixed_blockers` from a fresh assessment of it.

**Requirements**

- `survey_fixed` must have the same number of rows as `survey`.
- Do not modify `survey` itself.

**Hints**

- `counts = survey["region"].value_counts()` then `small = counts[counts < MIN_GROUP].index`.
- `Series.where(condition, other)` keeps the value where the condition holds and substitutes
  `other` where it does not — or use `np.where`.
- Assess the fixed frame exactly as in Task 1.

**Expected output.** Six regions, no blockers, then `Task 2 complete`.

In [ ]:
# TODO 2: combine the regions that are too small, then re-assess.
survey_fixed = None
fixed_report = None
fixed_blockers = None

if survey_fixed is not None:
    print(survey_fixed["region"].value_counts().to_string())
    if fixed_report is not None:
        print()
        print_assessment(fixed_report, "household survey, regions combined")

In [ ]:
# Verification for Task 2.
reference_counts = survey["region"].value_counts()
small_regions = set(reference_counts[reference_counts < MIN_GROUP].index)
problems = []

if survey_fixed is None:
    problems.append("survey_fixed is still None")
elif not isinstance(survey_fixed, pd.DataFrame):
    problems.append(f"survey_fixed should be a DataFrame, got {type(survey_fixed).__name__}")
else:
    if len(survey_fixed) != len(survey):
        problems.append(f"expected {len(survey)} rows, got {len(survey_fixed)}; "
                        "combine the small regions, do not drop them")
    fixed_counts = survey_fixed["region"].value_counts()
    if small_regions & set(fixed_counts.index):
        problems.append(f"these regions should have been relabelled: "
                        f"{sorted(small_regions & set(fixed_counts.index))}")
    elif "Other" not in fixed_counts.index:
        problems.append("no 'Other' category — the small regions must go somewhere")
    elif int(fixed_counts.min()) < MIN_GROUP:
        problems.append(f"the smallest group is still {int(fixed_counts.min())} rows")
    if set(survey["region"].unique()) != set(reference_counts.index):
        problems.append("survey itself was modified — work on a copy")

    if fixed_report is None or fixed_blockers is None:
        problems.append("fixed_report or fixed_blockers is still None")
    elif list(fixed_blockers) != list(fixed_report.get("blockers", ["x"])):
        problems.append("fixed_blockers should be the report's 'blockers' list")
    elif len(fixed_blockers) != 0:
        problems.append(f"there are still blockers: {fixed_blockers}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 2 complete. {survey['region'].nunique()} regions became "
          f"{survey_fixed['region'].nunique()}, and the verdict is "
          f"'{fixed_report['verdict']}'.")
    print("Every household is still in the dataset. 'Other' is an honest category as long as")
    print("you say in the report which regions are in it.")

### Task 3 — Is the question answerable?

The survey now passes the structural checks. That is not the same as being able to detect the
difference the student cares about, which is **$10 a month** between regions.

**Problem.** Using the two quantities computed in the cell below, assign:

| Variable | Contents |
|---|---|
| `survey_mde` | the smallest detectable difference in spending, at the size of the smallest region |
| `n_needed` | the smallest `n` per group at which the MDE would be at most $10 |

Both come from the survey with its small regions combined, so this task works whether or
not Task 2 is finished.

**Hints**

- `minimum_detectable_effect(sd, n)` with the defaults (alpha 0.05, power 0.80).
- For `n_needed`, loop upward from 2 until the MDE drops to 10 or below, or rearrange the
  formula. Either is accepted.
- `n_needed` must be a whole number.

**Expected output.** Two numbers, then `Task 3 complete`.

In [ ]:
# Computed here from the survey with its small regions combined, so that this task
# works whether or not Task 2 is finished.
_counts = survey["region"].value_counts()
_combined = survey["region"].where(
    ~survey["region"].isin(_counts[_counts < MIN_GROUP].index), "Other")
_regrouped = survey.assign(region=_combined)

survey_sd = _regrouped.groupby("region")["spend_usd"].std(ddof=1).mean()
smallest_region = int(_combined.value_counts().min())

print(f"within-region sd        {survey_sd:.2f} USD")
print(f"smallest region         {smallest_region} households")
print(f"difference of interest  $10.00")

In [ ]:
# TODO 3: what can this survey detect, and what would it take to detect $10?
survey_mde = None
n_needed = None

if None not in (survey_mde, n_needed):
    print(f"smallest detectable difference   ${survey_mde:.2f}")
    print(f"households per group needed for $10   {n_needed}")

In [ ]:
# Verification for Task 3.
expected_mde = minimum_detectable_effect(survey_sd, smallest_region)
expected_n = next(n for n in range(2, 100000)
                  if minimum_detectable_effect(survey_sd, n) <= 10.0)
problems = []

if None in (survey_mde, n_needed):
    problems.append("survey_mde or n_needed is still None")
else:
    if not np.isclose(survey_mde, expected_mde, rtol=1e-6):
        if np.isclose(survey_mde, minimum_detectable_effect(survey_sd, len(survey_fixed))):
            problems.append("survey_mde used the whole sample; use the SMALLEST group")
        else:
            problems.append(f"survey_mde should be {expected_mde:.4f}, got {survey_mde:.4f}")
    if int(n_needed) != expected_n:
        if abs(int(n_needed) - expected_n) <= 1:
            problems.append(f"n_needed is {n_needed}; it should be {expected_n} — "
                            "check whether you rounded up")
        else:
            problems.append(f"n_needed should be {expected_n}, got {n_needed}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 3 complete. This survey can detect about ${survey_mde:.0f}; "
          f"${10:.0f} would need {n_needed} households per region.")
    print("The smallest region has far fewer than that, so a $10 difference involving it")
    print("would not be detected. The question has to change, or the regions do.")

### Task 4 — Your own project

This is the task the session exists for.

**Problem.** Fill in `MY_PROJECT` with your own project, following the shape of `example_plan`
in section 2.4, and make `validate_plan(MY_PROJECT)` return an empty list.

**Requirements** — enforced by the checker:

- `question` ends in `?` and is at least eight words.
- `decision` says what somebody **would do** differently.
- `rows` is at least 500 — the real row count of your real dataset.
- `risks` has at least two entries.
- `source` is a URL, or says how the data were collected.

**If you do not yet have a dataset**, use the clinic data from section 2.2 and write the plan
around it. You will replace it before checkpoint 2, and the practice is the point.

**Expected output.** `Task 4 complete`, and a plan you can start from tomorrow.

In [ ]:
# TODO 4: your project plan. Replace every value.
MY_PROJECT = {
    "question": "",
    "decision": "",
    "dataset": "",
    "source": "",
    "outcome": "",
    "comparison": "",
    "rows": 0,
    "risks": [],
}

issues = validate_plan(MY_PROJECT)
if issues:
    print("Not ready yet:")
    for issue in issues:
        print("  -", issue)
else:
    for key, value in MY_PROJECT.items():
        print(f"  {key:<12} {value}")

In [ ]:
# Verification for Task 4.
problems = validate_plan(MY_PROJECT)

if not problems:
    # a plan that is merely well-formed is not necessarily yours
    if MY_PROJECT["question"].strip() == example_plan["question"].strip():
        problems.append("that is the worked example's question — write your own")
    if MY_PROJECT["source"] == example_plan["source"]:
        problems.append("that is the worked example's source")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print("Task 4 complete. Your plan passes every check.")
    print(f"\n  Question:  {MY_PROJECT['question']}")
    print(f"  Decision:  {MY_PROJECT['decision']}")
    print(f"  Risks:     {len(MY_PROJECT['risks'])} named")
    print("\nShow it to the person next to you and ask them to attack the decision sentence.")

---

# Part 4 — Homework

Two functions, both of which you will use in the project itself. Then the checkpoint-1
deliverable.

---

## Problem 1 — The summary table your report needs

**Problem.** Write `summarise_for_report(df, value_col, group_col, level=0.95)`.

It returns a **DataFrame** with one row per group, indexed by the group label, with exactly
these columns in this order:

| Column | Contents |
|---|---|
| `n` | number of **non-missing** values in the group |
| `mean` | the mean |
| `sd` | the standard deviation, `ddof=1` |
| `se` | the standard error of the mean |
| `ci_low`, `ci_high` | the `level` confidence interval for the mean, from the t distribution |

Sorted by `mean`, **descending**.

**Requirements**

- Drop missing values within each group before computing anything.
- Raise `TypeError` if `df` is not a DataFrame, or if `value_col` is not numeric.
- Raise `KeyError` if either column is absent.
- Raise `ValueError` if `level` is not strictly between 0 and 1, or if **any** group has fewer
  than two non-missing values — a standard error needs at least two.
- Do not modify the input.

**Hints**

- `stats.t.ppf(0.5 + level / 2, df=n - 1)` is the critical value; this is Week 12, Task 2.
- Build a list of dictionaries and pass it to `pd.DataFrame`, then `set_index` and
  `sort_values`.
- `df.groupby(group_col)[value_col]` gives one Series per group.

In [ ]:
# Problem 1 — starter code.

def summarise_for_report(df, value_col: str, group_col: str, level: float = 0.95):
    """One row per group: n, mean, sd, se and a confidence interval for the mean.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    value_col, group_col : str
        Column names.
    level : float
        Confidence level, strictly between 0 and 1.

    Returns
    -------
    pandas.DataFrame
        Indexed by group label, columns [n, mean, sd, se, ci_low, ci_high],
        sorted by mean descending.

    Raises
    ------
    TypeError, KeyError, ValueError
        As described in the problem statement.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete summarise_for_report()")


print(summarise_for_report(visits.dropna(subset=["wait_minutes"]),
                           "wait_minutes", "department").round(3).to_string())

In [ ]:
# Problem 1 — assertions. Run until this prints "all assertions passed".

# Designed so every number can be checked on paper.
#   P: 10 12 14 16 18   mean 14, sd sqrt(10), se sqrt(2)
#   Q: 20 22 24 26 28   mean 24, same spread
#   R: 30, NaN, 34      mean 32, sd sqrt(8),  se 2
check = pd.DataFrame({
    "team": ["P"] * 5 + ["Q"] * 5 + ["R"] * 3,
    "score": [10.0, 12.0, 14.0, 16.0, 18.0,
              20.0, 22.0, 24.0, 26.0, 28.0,
              30.0, np.nan, 34.0],
    "label": ["x"] * 13,
})

out = summarise_for_report(check, "score", "team")

assert isinstance(out, pd.DataFrame), f"Must return a DataFrame; got {type(out).__name__}"
assert list(out.columns) == ["n", "mean", "sd", "se", "ci_low", "ci_high"], (
    f"Wrong columns or order: {list(out.columns)}"
)
assert list(out.index) == ["R", "Q", "P"], (
    f"Sort by mean DESCENDING: expected ['R', 'Q', 'P'], got {list(out.index)}"
)

assert list(out["n"]) == [2, 5, 5], (
    f"n counts NON-MISSING values: expected [2, 5, 5], got {list(out['n'])}"
)
assert np.allclose(out["mean"].to_numpy(dtype=float), [32.0, 24.0, 14.0])
assert np.allclose(out["sd"].to_numpy(dtype=float),
                   [np.sqrt(8.0), np.sqrt(10.0), np.sqrt(10.0)]), (
    f"sd must use ddof=1; got {list(out['sd'])}"
)
assert np.allclose(out["se"].to_numpy(dtype=float),
                   [2.0, np.sqrt(2.0), np.sqrt(2.0)]), (
    f"se is sd / sqrt(n); got {list(out['se'])}"
)

# the interval, checked against the formula
for group, n, mean, se in [("R", 2, 32.0, 2.0), ("Q", 5, 24.0, np.sqrt(2.0)),
                           ("P", 5, 14.0, np.sqrt(2.0))]:
    margin = stats.t.ppf(0.975, df=n - 1) * se
    assert np.isclose(out.loc[group, "ci_low"], mean - margin), (
        f"{group}: ci_low should be {mean - margin:.6f}; got {out.loc[group, 'ci_low']:.6f}"
    )
    assert np.isclose(out.loc[group, "ci_high"], mean + margin)

# a different level gives a wider interval
wide = summarise_for_report(check, "score", "team", level=0.99)
assert wide.loc["P", "ci_low"] < out.loc["P", "ci_low"], "A 99% interval must be wider"
assert wide.loc["P", "ci_high"] > out.loc["P", "ci_high"]

# --- errors ---
for bad in ["not a frame", check["score"], None]:
    try:
        summarise_for_report(bad, "score", "team")
    except TypeError:
        pass
    else:
        raise AssertionError(f"{type(bad).__name__} input must raise TypeError")

for bad_value, bad_group in [("nope", "team"), ("score", "nope")]:
    try:
        summarise_for_report(check, bad_value, bad_group)
    except KeyError:
        pass
    else:
        raise AssertionError(f"Missing column ({bad_value}, {bad_group}) must raise KeyError")

try:
    summarise_for_report(check, "label", "team")
except TypeError:
    pass
else:
    raise AssertionError("A non-numeric value_col must raise TypeError")

for bad_level in [0.0, 1.0, 95, -0.5]:
    try:
        summarise_for_report(check, "score", "team", level=bad_level)
    except ValueError:
        pass
    else:
        raise AssertionError(f"level={bad_level} must raise ValueError")

lonely = pd.DataFrame({"team": ["P", "P", "S"], "score": [1.0, 2.0, 3.0]})
try:
    summarise_for_report(lonely, "score", "team")
except ValueError:
    pass
else:
    raise AssertionError("A group with fewer than two values must raise ValueError")

# --- the input is untouched ---
before = check.copy()
_ = summarise_for_report(check, "score", "team")
assert check.equals(before), "summarise_for_report must not modify its input"

# --- and it works on the notebook's own data ---
real = summarise_for_report(visits.dropna(subset=["wait_minutes"]),
                            "wait_minutes", "session")
assert list(real.index) == ["afternoon", "morning"], (
    f"Afternoons wait longer, so they sort first; got {list(real.index)}"
)
assert (real["ci_low"] < real["mean"]).all() and (real["mean"] < real["ci_high"]).all()

print("Problem 1: all assertions passed.")

---

## Problem 2 — How much data would you need?

Section 2.3 asked what a given sample can detect. Scoping usually needs the question the other
way round: *I care about a difference of this size — how many observations do I need?*

**Problem.** Write `required_sample_size(sd, target_difference, alpha=0.05, power=0.80)`.

Rearranging the formula in `minimum_detectable_effect`:

$$n \;=\; 2\left(\frac{(z_{1-\alpha/2} + z_{\text{power}})\,s}{\Delta}\right)^{2}$$

and rounded **up** to a whole number, because you cannot recruit 43.2 people.

**Requirements**

- Return an `int`, always rounded up.
- Raise `ValueError` if `sd` or `target_difference` is not strictly positive, or if `alpha` or
  `power` is not strictly between 0 and 1.
- The result must be **consistent** with `minimum_detectable_effect`: feeding it back must give
  a detectable effect no larger than `target_difference`.

**Hints**

- `stats.norm.ppf(1 - alpha / 2)` and `stats.norm.ppf(power)`.
- `int(np.ceil(...))`.
- Check the round trip yourself before running the assertions.

In [ ]:
# Problem 2 — starter code.

def required_sample_size(sd, target_difference, alpha=0.05, power=0.80) -> int:
    """Observations per group needed to detect a difference of `target_difference`.

    Parameters
    ----------
    sd : float
        Within-group standard deviation of the outcome. Must be positive.
    target_difference : float
        The smallest difference worth detecting, in the units of the outcome.
        Must be positive.
    alpha, power : float
        Significance level and desired power, each strictly between 0 and 1.

    Returns
    -------
    int
        Observations per group, rounded up.

    Raises
    ------
    ValueError
        If any argument is out of range.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete required_sample_size()")


for target in [20.0, 10.0, 5.0, 2.0]:
    n = required_sample_size(survey_sd, target)
    print(f"  to detect ${target:5.2f}:  {n:6,d} households per group")

In [ ]:
# Problem 2 — assertions. Run until this prints "all assertions passed".

multiplier = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)

# exact, from the formula
for sd, target in [(10.0, 5.0), (26.0, 10.0), (1.0, 0.2), (3.5, 3.5)]:
    expected = int(np.ceil(2.0 * ((multiplier * sd) / target) ** 2))
    got = required_sample_size(sd, target)
    assert got == expected, f"sd={sd}, target={target}: expected {expected}, got {got}"

assert isinstance(required_sample_size(10.0, 5.0), (int, np.integer)), "Must return an int"

# always rounded UP, never down or to nearest
exact = 2.0 * ((multiplier * 10.0) / 5.0) ** 2
assert required_sample_size(10.0, 5.0) >= exact, (
    f"Round UP: the exact value is {exact:.4f}"
)
assert required_sample_size(10.0, 5.0) - exact < 1.0, "Round up, not up by a whole extra unit"

# a smaller target needs more data, and it scales with the square
assert required_sample_size(10.0, 2.5) > required_sample_size(10.0, 5.0)
assert abs(required_sample_size(10.0, 2.5) / required_sample_size(10.0, 5.0) - 4.0) < 0.05, (
    "Halving the target should roughly quadruple the sample"
)

# more power and a stricter alpha both cost data
assert required_sample_size(10.0, 5.0, power=0.90) > required_sample_size(10.0, 5.0)
assert required_sample_size(10.0, 5.0, alpha=0.01) > required_sample_size(10.0, 5.0)

# --- the round trip: it must agree with minimum_detectable_effect ---
for sd, target in [(26.0, 10.0), (12.0, 3.0), (100.0, 9.0), (0.4, 0.05)]:
    n = required_sample_size(sd, target)
    achieved = minimum_detectable_effect(sd, n)
    assert achieved <= target + 1e-9, (
        f"sd={sd}, target={target}: n={n} only detects {achieved:.4f}"
    )
    if n > 3:
        assert minimum_detectable_effect(sd, n - 1) > target, (
            f"sd={sd}, target={target}: n={n} is larger than necessary"
        )

# --- errors ---
for bad_sd in [0.0, -1.0]:
    try:
        required_sample_size(bad_sd, 5.0)
    except ValueError:
        pass
    else:
        raise AssertionError(f"sd={bad_sd} must raise ValueError")

for bad_target in [0.0, -2.0]:
    try:
        required_sample_size(10.0, bad_target)
    except ValueError:
        pass
    else:
        raise AssertionError(f"target_difference={bad_target} must raise ValueError")

for bad in [0.0, 1.0, 1.5, -0.1]:
    for keyword in ["alpha", "power"]:
        try:
            required_sample_size(10.0, 5.0, **{keyword: bad})
        except ValueError:
            pass
        else:
            raise AssertionError(f"{keyword}={bad} must raise ValueError")

print("Problem 2: all assertions passed.")

---

## Checkpoint 1 — due with this homework

Submit, in your repository:

1. **`final_project.ipynb`**, generated by `write_project_template` with your name, with
   sections 1 and 2 filled in.
2. **Your `MY_PROJECT` plan**, passing `validate_plan` with no issues.
3. **The output of `assess_dataset`** on your real dataset, pasted into section 2 of the
   project notebook, with a sentence on any warnings.
4. **The output of `required_sample_size`** for the difference you care about, and one sentence
   on whether your dataset reaches it.

Point 4 is the one that saves people. If the number comes back larger than your dataset, say so
**now**, and change the question while changing it is still cheap.

## Written question

Answer in the Markdown cell below, in 150–250 words.

1. State your question, and the sentence *"depending on the answer, somebody would ..."*.
2. Name the single biggest threat to your project being finished on time, and what you will do
   about it this week.
3. Week 13 showed a study that could not detect the effect that was actually present. Using
   your own numbers from `required_sample_size`, say what size of effect your project can
   detect, and what you will write in the report if you find something smaller.

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] `final_project.ipynb` exists in your repository with sections 1 and 2 written.
- [ ] `MY_PROJECT` passes `validate_plan`.
- [ ] The written question is answered in 150–250 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 15

Week 15 is review and presentations. Bring:

- your project notebook, in whatever state it is in — a working draft beats a polished plan,
- three figures you might show,
- and your findings paragraph, even if you are not happy with it.

The session covers how to present an analysis in ten minutes, a self-check you can run on your
own notebook before submitting, and a revision map across the whole course.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.

Pfaffelhuber, P. *Python for Data Analysis*. §10.
Available at `https://pfaffelh.github.io/python_for_data/`.

Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.). Lawrence
Erlbaum Associates.